# Etapa 3b — Mejoramiento y restauración: comparación ampliada

*Complementa a `03_mejoramiento.ipynb` (Karolina): agrega Wiener contra el desenfoque, el suavizado de bloques de compresión, las cadenas combinadas y la validación cruzada. Antes se llamaba `03_mejoramiento.ipynb` y usaba `src/enhance.py`; se renombraron para no chocar con los archivos de Karolina y Leah, sin cambiar el código.*

**Pregunta.** ¿Qué preprocesamiento conviene aplicar a la tercera componente (mapa principal) y al verde (referencia clásica) antes de los filtros de realce? Solo desarrollo.

**Candidatas** (`src/restoration.py`): sin mejora; corrección del fondo (mediana grande restada); ecualización adaptativa del histograma; filtro homomórfico; Wiener contra el ruido (adaptativo local); Wiener contra el desenfoque (deconvolución con desenfoque gaussiano supuesto, 2 anchos × 2 niveles de ruido); suavizado de bloques de compresión (sobre la foto a color a resolución original, antes de calcular los mapas). En una segunda pasada, el mejor mejoramiento de cada mapa combinado con las restauraciones y la combinación clásica corrección del fondo + ecualización.

**Cómo se mide.** Se calcula a la mitad de la resolución (`STAGE3_FACTOR`), se agranda el mapa y se evalúa a resolución original dentro del campo de visión de la base: área bajo la curva de todos los vasos y de los vasos finos (ancho marcado ≤ 3,47) contra todo el fondo. El sentido de cada candidata (vasos claros u oscuros) se fija con la media de desarrollo.

**Cómo se elige.** Por mapa: la más simple entre las no inferiores a la mejor en el área de todos los vasos (margen −0,02, remuestreo de personas). Orden de simplicidad: sin mejora, fondo, ecualización, homomórfico, fondo + ecualización, cadenas con restauración. Wiener y el suavizado de bloques son temas obligatorios: se adoptan si la cadena con ellos es no inferior a la misma sin ellos (esquema 5.4). Control: validación cruzada por personas de la elección.


In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from PIL import Image
from skimage.transform import resize

from src import config as C
from src.color import color_maps
from src.data import dev_ids, load_fov, load_image, load_split, load_vessels
from src.restoration import (background_correct, clahe, deblock, homomorphic, wiener_deblur,
                         wiener_denoise)
from src.fov import detect_fov_robust, fov_diameter
from src.metrics import auc_by_subset
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means
from src.vessels import width_map

ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
F = C.STAGE3_FACTOR
MAPS = ["PC3", "Verde"]
DEBLUR = [(s, b) for s in C.WIENER_DEBLUR_SIGMAS for b in C.WIENER_DEBLUR_BALANCES]

def half(rgb):
    return np.asarray(Image.fromarray(rgb).reduce(F))

def up(m, shape):
    return resize(m, shape, order=1, preserve_range=True, anti_aliasing=False).astype(np.float32)

def enhancements(m, region, d):
    size = int(round(C.BACKGROUND_WINDOW_REL * d)) | 1
    return {"fondo": background_correct(m, region, size),
            "ecualización": clahe(m, region, C.CLAHE_CLIP, C.CLAHE_KERNEL_FRAC),
            "homomórfico": homomorphic(m, region, C.HOMOMORPHIC_SIGMA_REL * d),
            "fondo + ecualización": clahe(background_correct(m, region, size), region,
                                          C.CLAHE_CLIP, C.CLAHE_KERNEL_FRAC)}

def restorations(m, region):
    out = {"Wiener ruido": wiener_denoise(m, region, C.WIENER_SIZE)}
    for s, b in DEBLUR:
        out[f"Wiener desenfoque σ={s:g} b={b:g}"] = wiener_deblur(m, region, s, b)
    return out

def prepare(image_id):
    rgb, fov, ves = load_image(image_id), load_fov(image_id), load_vessels(image_id)
    thin = (width_map(ves & fov) <= C.WIDTH_LIMITS_PX[0]) & ves & fov
    small, small_db = half(rgb), half(deblock(rgb))
    region = detect_fov_robust(small)
    return rgb.shape[:2], fov, ves, thin, small, small_db, region, fov_diameter(region)

## Pasada 1: cada candidata sola

In [2]:
rows, cache_thin = [], {}
for image_id in ids:
    shape, fov, ves, thin, small, small_db, region, d = prepare(image_id)
    base = color_maps(small, region)
    blocked = color_maps(small_db, region)
    for name in MAPS:
        m = base[name]
        cands = {"sin mejora": m, **enhancements(m, region, d), **restorations(m, region),
                 "bloques": blocked[name]}
        cands = {k: v for k, v in cands.items() if k != "fondo + ecualización"}
        for cand, x in cands.items():
            a = auc_by_subset(up(x, shape), ves, {"finos": thin}, fov)
            rows.append({"image": image_id, "map": name, "cand": cand, "todos": a["todos"], "finos": a["finos"]})
    print(image_id, end=" ", flush=True)
P1 = pd.DataFrame(rows)
P1.to_csv(ROOT / "results" / "03_pasada1.csv", index=False)

01_h 

02_h 

03_h 

04_h 

07_h 

08_h 

... (20 outputs left out)

11_g 

12_g 

13_g 

14_g 

In [3]:
def orient(frame):
    """Sentido por candidata (columna) con la media de desarrollo."""
    flip = frame.mean() < 0.5
    out = frame.copy()
    out.loc[:, flip] = 1 - frame.loc[:, flip]
    return out

def table(P, mapname, measure="todos"):
    w = orient(P[P["map"] == mapname].pivot(index="image", columns="cand", values=measure))
    return w

def summary(P, mapname):
    w, wf = table(P, mapname), table(P, mapname, "finos")
    diffs = w.sub(w["sin mejora"], axis=0)
    lo, hi = percentile_interval(person_bootstrap_means(diffs, dev["person"]))
    return pd.DataFrame({"área (todos)": w.mean(), "área (finos)": wf.mean(), "peor imagen": w.min(),
                         "vs sin mejora": diffs.mean(), "límite inferior": lo, "límite superior": hi}
                        ).sort_values("área (todos)", ascending=False)

for name in MAPS:
    print(f"\n=== {name} ===")
    print(summary(P1, name).round(4).to_string())


=== PC3 ===
                              área (todos)  área (finos)  peor imagen  vs sin mejora  límite inferior  límite superior
cand                                                                                                                  
ecualización                        0.8589        0.6641       0.7848         0.0386           0.0321           0.0455
homomórfico                         0.8575        0.6748       0.7769         0.0372           0.0317           0.0427
fondo                               0.8454        0.6921       0.7497         0.0251           0.0143           0.0360
Wiener desenfoque σ=2 b=0.1         0.8271        0.6531       0.7371         0.0068           0.0032           0.0103
Wiener desenfoque σ=1 b=0.1         0.8242        0.6345       0.7251         0.0038           0.0031           0.0046
Wiener ruido                        0.8228        0.6190       0.7162         0.0025           0.0005           0.0051
bloques                            

## Pasada 2: el mejor mejoramiento de cada mapa combinado con las restauraciones

In [4]:
best_enh, best_deblur = {}, {}
for name in MAPS:
    w = table(P1, name).mean()
    best_enh[name] = w[["fondo", "ecualización", "homomórfico"]].idxmax()
    best_deblur[name] = w[[c for c in w.index if c.startswith("Wiener desenfoque")]].idxmax()
print("Mejor mejoramiento:", best_enh)
print("Mejor deconvolución:", best_deblur)

def chain(m, region, d, restoration, enh):
    size = int(round(C.BACKGROUND_WINDOW_REL * d)) | 1
    if restoration == "Wiener ruido":
        m = wiener_denoise(m, region, C.WIENER_SIZE)
    elif restoration.startswith("Wiener desenfoque"):
        s = float(restoration.split("σ=")[1].split()[0]); b = float(restoration.split("b=")[1])
        m = wiener_deblur(m, region, s, b)
    return enhancements(m, region, d)[enh]

rows = []
for image_id in ids:
    shape, fov, ves, thin, small, small_db, region, d = prepare(image_id)
    base = color_maps(small, region)
    blocked = color_maps(small_db, region)
    for name in MAPS:
        e = best_enh[name]
        cands = {"fondo + ecualización": enhancements(base[name], region, d)["fondo + ecualización"]}
        for r in ("Wiener ruido", best_deblur[name]):
            label = "Wiener ruido" if r == "Wiener ruido" else "Wiener desenfoque"
            cands[f"{label} → {e}"] = chain(base[name], region, d, r, e)
            cands[f"bloques → {label} → {e}"] = chain(blocked[name], region, d, r, e)
        cands[f"bloques → {e}"] = enhancements(blocked[name], region, d)[e]
        for cand, x in cands.items():
            a = auc_by_subset(up(x, shape), ves, {"finos": thin}, fov)
            rows.append({"image": image_id, "map": name, "cand": cand, "todos": a["todos"], "finos": a["finos"]})
    print(image_id, end=" ", flush=True)
P2 = pd.DataFrame(rows)
P2.to_csv(ROOT / "results" / "03_pasada2.csv", index=False)
P = pd.concat([P1, P2], ignore_index=True)

Mejor mejoramiento: {'PC3': 'ecualización', 'Verde': 'fondo'}
Mejor deconvolución: {'PC3': 'Wiener desenfoque σ=2 b=0.1', 'Verde': 'Wiener desenfoque σ=2 b=0.01'}


01_h 

02_h 

03_h 

04_h 

07_h 

... (21 outputs left out)

11_g 

12_g 

13_g 

14_g 

In [5]:
SIMPLE = ["sin mejora", "fondo", "ecualización", "homomórfico", "fondo + ecualización"]
choice = {}
for name in MAPS:
    w = table(P, name)
    best = w.mean().idxmax()
    diffs = w.sub(w[best], axis=0)
    lo, hi = percentile_interval(person_bootstrap_means(diffs, dev["person"]))
    res = pd.DataFrame({"área": w.mean(), "vs mejor": diffs.mean(), "límite inferior": lo,
                        "no inferior": lo >= -C.NONINF_MARGIN_AUC, "área finos": table(P, name, "finos").mean()}
                       ).sort_values("área", ascending=False)
    order = SIMPLE + [c for c in res.index if c not in SIMPLE]
    choice[name] = next(c for c in order if c in res.index and res.loc[c, "no inferior"])
    print(f"\n=== {name}: mejor {best}; elegida (la más simple no inferior): {choice[name]} ===")
    print(res.round(4).to_string())


=== PC3: mejor Wiener ruido → ecualización; elegida (la más simple no inferior): ecualización ===
                                              área  vs mejor  límite inferior  no inferior  área finos
cand                                                                                                  
Wiener ruido → ecualización                 0.8689    0.0000           0.0000         True      0.6690
bloques → Wiener ruido → ecualización       0.8684   -0.0005          -0.0008         True      0.6671
bloques → ecualización                      0.8601   -0.0088          -0.0128         True      0.6641
ecualización                                0.8589   -0.0100          -0.0142         True      0.6641
homomórfico                                 0.8575   -0.0115          -0.0180         True      0.6748
bloques → Wiener desenfoque → ecualización  0.8536   -0.0153          -0.0219        False      0.6837
Wiener desenfoque → ecualización            0.8535   -0.0154          -0.0219

**Temas obligatorios:** la cadena con cada restauración contra la misma sin ella (mismo mejoramiento).

In [6]:
rows = []
for name in MAPS:
    w = table(P, name)
    e = best_enh[name]
    for with_r in [f"Wiener ruido → {e}", f"Wiener desenfoque → {e}", f"bloques → {e}"]:
        diff = w[with_r] - w[e]
        lo, hi = percentile_interval(person_bootstrap_means(diff, dev["person"]))
        rows.append({"mapa": name, "con restauración": with_r, "diferencia": diff.mean(),
                     "límite inferior": lo[0], "límite superior": hi[0], "se adopta": lo[0] >= -C.NONINF_MARGIN_AUC})
pd.DataFrame(rows).round(4)

,mapa,con restauración,diferencia,límite inferior,límite superior,se adopta
0,PC3,Wiener ruido → ecualización,0.0100,0.0067,0.0142,True
1,PC3,Wiener desenfoque → ecualización,-0.0054,-0.0093,-0.0015,True
2,PC3,bloques → ecualización,0.0012,0.0009,0.0016,True
3,Verde,Wiener ruido → fondo,0.0011,-0.0001,0.0021,True
4,Verde,Wiener desenfoque → fondo,-0.0346,-0.0396,-0.0295,False
5,Verde,bloques → fondo,0.0006,0.0005,0.0008,True


**Suavizado de bloques por nivel de compresión** (debería ayudar más en las imágenes más comprimidas):

In [7]:
rows = []
for name in MAPS:
    w = table(P, name)
    e = best_enh[name]
    d = (w[f"bloques → {e}"] - w[e]).to_frame("dif").join(dev["more_compressed"])
    for level, sub in d.groupby("more_compressed"):
        rows.append({"mapa": name, "más comprimidas": level, "imágenes": len(sub), "diferencia media": sub["dif"].mean()})
pd.DataFrame(rows).round(4)

,mapa,más comprimidas,imágenes,diferencia media
0,PC3,False,18,0.0011
1,PC3,True,12,0.0014
2,Verde,False,18,0.0007
3,Verde,True,12,0.0005


**Estabilidad de la elección** (validación cruzada por personas, 5 grupos). En cada vuelta se fija el sentido y se elige la más simple no inferior con las personas de entrenamiento; se mide en las de afuera. *El mejor mejoramiento y la mejor deconvolución de la pasada 2 se eligieron con todo desarrollo.*

In [8]:
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
for name in MAPS:
    raw = P[P["map"] == name].pivot(index="image", columns="cand", values="todos")
    picks, out = [], []
    for k in range(5):
        tr, te = folds.index[folds != k], folds.index[folds == k]
        flip = raw.loc[tr].mean() < 0.5
        o = raw.copy(); o.loc[:, flip] = 1 - raw.loc[:, flip]
        best = o.loc[tr].mean().idxmax()
        diffs = o.loc[tr].sub(o.loc[tr, best], axis=0)
        lo, _ = percentile_interval(person_bootstrap_means(diffs, dev.loc[tr, "person"]))
        ok = pd.Series(lo >= -C.NONINF_MARGIN_AUC, index=diffs.columns)
        order = SIMPLE + [c for c in o.columns if c not in SIMPLE]
        pick = next(c for c in order if c in ok.index and ok[c])
        picks.append(pick); out.append(o.loc[te, pick])
    print(f"{name}: elegidas por vuelta {picks}; área afuera {pd.concat(out).mean():.4f}")

PC3: elegidas por vuelta ['ecualización', 'ecualización', 'ecualización', 'ecualización', 'ecualización']; área afuera 0.8589


Verde: elegidas por vuelta ['fondo', 'fondo', 'fondo', 'fondo', 'fondo']; área afuera 0.9407


## Decisión (escrita después de correr)

**Tercera componente.** La más simple entre las no inferiores es la **ecualización adaptativa** (0,859; la mejor, Wiener contra el ruido seguido de ecualización, da 0,869). La corrección del fondo le suma poco (0,845).

**Verde.** La **corrección del fondo** lo lleva de 0,687 a **0,941** (vasos finos de 0,570 a 0,855). Es lo que más mejora en toda la etapa.

**Temas obligatorios** (se adoptan si la cadena con ellos no es inferior a la misma sin ellos):
- **Wiener contra el ruido: se adopta** en los dos mapas. Mejora a la tercera componente en 0,010 (0,007 a 0,014) y deja igual al verde (+0,001).
- **Wiener contra el desenfoque: no se adopta.** Empeora al verde en 0,035 y a la tercera componente en 0,005. Se informa como probado: con el desenfoque supuesto, deshacerlo amplifica más ruido del que recupera de nitidez.
- **Suavizado de bloques: se adopta** en los dos mapas (+0,001; algo más en las imágenes más comprimidas para la tercera componente).

**Cadenas resultantes:** tercera componente → bloques, Wiener contra el ruido, ecualización (0,868); verde → bloques, Wiener contra el ruido, corrección del fondo (0,942). La validación cruzada elige lo mismo en las 5 vueltas.

**Hallazgo que cambia la etapa 2.** Con su preprocesamiento, el verde (0,942) supera ampliamente a la tercera componente (0,868), también en vasos finos (0,846 contra 0,667). La ventaja de la tercera componente en la etapa 2 venía de comparar mapas crudos: el verde crudo separa mal por la iluminación despareja, no por falta de información. La elección del mapa principal y de los complementarios se rehace con la corrección del fondo aplicada a los diez mapas (`02d_mapas_corregidos.ipynb`). Se declara como desvío en el esquema: el mapa y el preprocesamiento se deciden juntos.
